In [1]:
import pickle
# 加载加载
with open("./pdf_documents.pkl", 'rb') as f:
    pdf_documents = pickle.load(f)
with open("./pdf_files.pkl","rb") as f:
    pdf_paths = pickle.load(f)
with open("./old_pdf_documents.pkl","rb") as f:
    old_pdf_documents = pickle.load(f)

/home/Glitterccc/projects/DKU_LLM/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# copy of pdf
pdf_new_documents = []

In [3]:
llama_parse_api_key = "llx-L0jVeG3SKiLhyHwkz5u1Yya90thvyzwhXutswUSW3qS7Usk9"
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
parser = LlamaParse(
    api_key=llama_parse_api_key,  # can also be set in your env as LLAMA_CLOUD_API_KEY
    result_type="markdown",  # "markdown" and "text" are available
    num_workers=4,  # if multiple files passed, split in `num_workers` API calls
    verbose=True,
    language="en",  # Optionally you can define a language, default=en
)

In [4]:
num = 0
for doc in pdf_documents:
    if len(doc) == 0:
        try:
            document = parser.load_data(pdf_paths[num])
        except:
            print(f"error when parse {pdf_paths[num]}")
        pdf_new_documents.append(document)
    else:
        pdf_new_documents.append(doc[0])
    num += 1

Error while parsing the file '/home/Glitterccc/projects/DKU_LLM/RAG_data/dku_website/ah.dukekunshan.edu.cn/sites/default/files/2023-04/eos_80d.pdf': Server disconnected without sending a response.
Started parsing the file under job_id f999d1bd-5aac-4712-aa32-af541d880b92
Error while parsing the file '/home/Glitterccc/projects/DKU_LLM/RAG_data/dku_website/ah.dukekunshan.edu.cn/sites/default/files/2023-04/pocket_cinema_camera_series.pdf': 'markdown'
Error while parsing the file '/home/Glitterccc/projects/DKU_LLM/RAG_data/dku_website/ah.dukekunshan.edu.cn/sites/default/files/2023-04/eos_200d.pdf': Server disconnected without sending a response.
Error while parsing the file '/home/Glitterccc/projects/DKU_LLM/RAG_data/dku_website/dnas.dukekunshan.edu.cn/wp-content/uploads/2024/06/MP_Hardware_Guide.pdf': Server disconnected without sending a response.
Started parsing the file under job_id cac11eca-a392-4bab-bf76-a40f56f6cedc


In [19]:
# 空白的index
blank_index = []
tmp = 0
for doc in pdf_documents:
    if len(doc) == 0:
        blank_index.append(tmp)
    tmp+=1
blank_index

[30, 35, 37, 44, 163]

In [28]:
# 向pdf_new_document中添加metadata
import re
for num in range(0,336):
    try:
        pdf_new_documents[num].metadata["file_path"] = pdf_paths[num]
        match = re.search(r'[^/]+\.pdf$', pdf_paths[num])
        file_name = match.group(0) if match else None
        pdf_new_documents[num].metadata["file_name"] = file_name
    except:
        print(num)

30
35
37
44


In [32]:
new_pdf_documents = []
for doc in pdf_new_documents:
    if isinstance(doc, list):
        continue
    else:
        new_pdf_documents.append(doc)

In [34]:
with open('tmp_documents.pkl', 'wb') as f:
    pickle.dump(new_pdf_documents, f)

In [110]:
def extract_after_rag_data(path):
    # 查找子字符串 "/RAG_data" 的索引
    index = path.find('/RAG_data')
    if index != -1:
        # 提取 "/RAG_data" 之后的部分
        return path[index + len('/RAG_data'):]
    else:
        return None

In [122]:
failed_match_documents = []
sum_new_document = []
from tqdm import tqdm
for new_document in tqdm(new_pdf_documents):
    have_equal_file = False
    for old_document in old_pdf_documents:
        if extract_after_rag_data(new_document.metadata["file_path"]) == extract_after_rag_data(old_document.metadata["file_path"]):
            have_equal_file = True
            sum_document = old_document
            sum_document.text = new_document.text
            sum_new_document.append(sum_document)
    if have_equal_file is False:
        failed_match_documents.append(new_document)


100%|██████████| 332/332 [00:00<00:00, 2068.86it/s]


In [49]:
# 新pdf document
# sum_new_document

In [144]:
len(sum_new_document)

328

In [145]:
with open('true_new_pdf_documents.pkl', 'wb') as f:
    pickle.dump(sum_new_document, f)